# 面试题：Unigram tokenizer 如何训练，为什么编码要用 Viterbi 而不是贪心？

## 面试回答主线

Unigram 把一句话看成若干子词独立生成的结果，切分概率是各 token 概率的乘积。训练通常从一个冗余候选词表开始，通过 EM 估计每个 token 的期望使用次数，再逐步裁剪对似然贡献较小的候选。推理是在所有合法切分路径中寻找总负对数概率最小的路径，因此需要 Viterbi 动态规划；“最长词片优先”不等于“概率最大”。与 BPE 的确定性合并历史相比，Unigram 天然保留多个可行切分，还能按概率采样做 subword regularization。单字符必须保底，否则裁剪后可能出现无路可走的输入。

## 真实案例：AI 学习内容站内搜索

下面用带搜索次数的中文查询日志训练小型 Unigram LM。数据字段与聚合日志一致，但内容和频次均为教学构造；小语料的分词只用于解释概率图和 EM，不能视为通用中文分词质量。

In [1]:
import math  # 导入对数与指数函数以计算路径代价和 EM 概率。
from collections import Counter  # 导入计数器以累计候选子串和期望 token 次数。
query_frequencies = [  # 构造带业务频次的 AI 学习搜索日志。
    ("机器学习", 180),  # 高频核心概念强化“机器”和“学习”。
    ("机器翻译", 95),  # 共享“机器”前缀并引入新后缀。
    ("学习资料", 85),  # 共享“学习”并连接内容类型。
    ("深度学习", 150),  # 高频组合强化“深度”和“学习”。
    ("深度搜索", 75),  # 共享“深度”并连接搜索任务。
    ("搜索引擎", 120),  # 高频产品概念形成候选长词片。
    ("引擎优化", 65),  # 共享“引擎”并引入优化后缀。
    ("机器视觉", 90),  # 共享“机器”并覆盖视觉领域。
]  # 结束训练日志列表。
print("查询         频次  字符数")  # 输出真实案例输入表标题。
for query, frequency in query_frequencies:  # 逐条展示查询、权重和基础长度。
    print(f"{query:<10} {frequency:>4} {len(query):>7}")  # 输出一条可读的聚合查询记录。
print("总搜索次数：", sum(frequency for _, frequency in query_frequencies))  # 输出 EM 统计所代表的总业务权重。

查询         频次  字符数
机器学习        180       4
机器翻译         95       4
学习资料         85       4
深度学习        150       4
深度搜索         75       4
搜索引擎        120       4
引擎优化         65       4
机器视觉         90       4
总搜索次数： 860


## Baseline：逐 character 编码

字符基线永远有路可走，却无法把高频语义片段压成一个 token。后续用同一批日志的“频次加权平均 token 数”比较压缩，同时用语料对数似然观察概率模型是否在学习。

In [2]:
def character_encode(text):  # 定义逐 Unicode 字符切分的保底基线。
    return list(text)  # 每个字符直接成为一个 token。
def weighted_average_length(rows, encoder):  # 定义按搜索次数加权的平均 token 长度指标。
    weighted_tokens = sum(len(encoder(text)) * frequency for text, frequency in rows)  # 汇总每条查询 token 数乘以业务频次。
    total_frequency = sum(frequency for _, frequency in rows)  # 计算所有查询次数作为分母。
    return weighted_tokens / total_frequency  # 返回随机一次搜索的平均 token 数。
baseline_average = weighted_average_length(query_frequencies, character_encode)  # 计算字符基线的统一比较指标。
print(f"字符基线加权平均 token 数：{baseline_average:.3f}")  # 输出后续 Unigram 结果的对照值。
for query, _ in query_frequencies:  # 展示每条训练查询的字符边界。
    print(f"{query:<10} -> {' | '.join(character_encode(query))}")  # 输出朴素切分以便观察冗余长度。

字符基线加权平均 token 数：4.000
机器学习       -> 机 | 器 | 学 | 习
机器翻译       -> 机 | 器 | 翻 | 译
学习资料       -> 学 | 习 | 资 | 料
深度学习       -> 深 | 度 | 学 | 习
深度搜索       -> 深 | 度 | 搜 | 索
搜索引擎       -> 搜 | 索 | 引 | 擎
引擎优化       -> 引 | 擎 | 优 | 化
机器视觉       -> 机 | 器 | 视 | 觉


## 核心实现一：候选词表与 forward-backward 软 EM

初始候选包含每条查询长度 1–4 的全部子串。E 步用 forward/backward 对所有切分路径求和，并计算每个 token 在后验分布下的期望次数；M 步归一化这些期望次数。这里用普通浮点数是因为句子很短，生产实现应在 log-space 计算以防下溢。

In [3]:
def build_candidates(rows, max_piece_length=4):  # 从加权查询中生成冗余候选词表及初始化计数。
    counts = Counter()  # 创建候选子串的加权出现次数计数器。
    characters = set()  # 收集必须永久保留的单字符集合。
    for text, frequency in rows:  # 遍历每条聚合查询及其业务权重。
        characters.update(text)  # 把全部基础字符加入保底集合。
        for start in range(len(text)):  # 枚举候选子串的起始位置。
            for end in range(start + 1, min(len(text), start + max_piece_length) + 1):  # 枚举不超过最大长度的右边界。
                counts[text[start:end]] += frequency  # 按查询频次累计当前候选子串的初始化权重。
    total = sum(counts.values())  # 计算全部候选权重作为概率归一化分母。
    probabilities = {piece: count / total for piece, count in counts.items()}  # 用加权子串频次初始化 Unigram 概率。
    return probabilities, characters  # 返回冗余候选概率与不可裁剪字符集合。
def forward_backward(text, probabilities, max_piece_length=4):  # 对一个查询计算总概率和各 token 后验期望次数。
    length = len(text)  # 保存字符长度以创建动态规划数组。
    forward = [0.0] * (length + 1)  # 创建从句首到各位置的路径概率和。
    forward[0] = 1.0  # 空前缀只有一条概率为一的路径。
    for start in range(length):  # 按拓扑顺序扩展每个可达前缀位置。
        for end in range(start + 1, min(length, start + max_piece_length) + 1):  # 枚举当前位置可取的候选片段。
            piece = text[start:end]  # 提取当前字符区间对应的候选 token。
            if piece in probabilities:  # 只有仍在词表中的候选才形成合法边。
                forward[end] += forward[start] * probabilities[piece]  # 累加经当前 token 到达右端点的路径概率。
    backward = [0.0] * (length + 1)  # 创建从各位置到句尾的路径概率和。
    backward[length] = 1.0  # 空后缀只有一条概率为一的路径。
    for start in range(length - 1, -1, -1):  # 按逆拓扑顺序计算每个后缀位置。
        for end in range(start + 1, min(length, start + max_piece_length) + 1):  # 枚举从当前位置出发的候选片段。
            piece = text[start:end]  # 提取当前动态规划边的 token 文本。
            if piece in probabilities:  # 跳过已经被裁剪的候选。
                backward[start] += probabilities[piece] * backward[end]  # 累加经当前 token 到达句尾的路径概率。
    partition = forward[length]  # 句尾 forward 值就是所有合法切分概率之和。
    expected = Counter()  # 创建当前查询的 token 后验期望次数计数器。
    for start in range(length):  # 再次遍历所有合法边计算其后验使用概率。
        for end in range(start + 1, min(length, start + max_piece_length) + 1):  # 枚举当前起点的候选 token。
            piece = text[start:end]  # 提取候选 token 文本。
            if piece in probabilities:  # 只有词表内 token 才参与后验计算。
                posterior = forward[start] * probabilities[piece] * backward[end] / partition  # 用前向、边概率和后向计算边后验。
                expected[piece] += posterior  # 累加该 token 在所有路径中的期望使用次数。
    return partition, expected, forward  # 返回句子概率、期望计数与可解释的前向数组。
initial_probabilities, mandatory_characters = build_candidates(query_frequencies)  # 创建冗余初始词表和保底字符。
focus_partition, focus_expected, focus_forward = forward_backward("机器学习", initial_probabilities)  # 对焦点查询执行一次完整 E 步。
print("初始候选数：", len(initial_probabilities), "，保底字符数：", len(mandatory_characters))  # 输出初始搜索空间规模。
print("‘机器学习’各前缀的概率和：", [f"{value:.6g}" for value in focus_forward])  # 展示 forward 动态规划的关键中间量。
print("‘机器学习’后验期望最高的 token：", [(piece, round(value, 4)) for piece, value in focus_expected.most_common(8)])  # 展示软 EM 并非只使用一条切分。

初始候选数： 59 ，保底字符数： 18
‘机器学习’各前缀的概率和： ['1', '0.0424419', '0.0442432', '0.0239535', '0.0251094']
‘机器学习’后验期望最高的 token： [('机器学习', 0.8336), ('机器', 0.0855), ('学习', 0.085), ('习', 0.046), ('机', 0.0407), ('机器学', 0.0402), ('器学习', 0.0354), ('学', 0.0041)]


## 核心实现二：EM 更新、保底字符与分阶段裁剪

每轮 E 步按查询频次累加期望次数，M 步重新归一化。第 3、5 轮裁剪低概率多字符候选，但永远保留所有单字符，因此 lattice 不会断路。裁剪会让对数似然短暂下降，这是用更小词表换取模型容量的真实代价。

In [4]:
def train_unigram(rows, initial, mandatory, iterations=8):  # 使用软 EM 和分阶段裁剪训练教学版 Unigram LM。
    probabilities = dict(initial)  # 复制初始概率以避免修改调用方状态。
    history = []  # 保存每轮似然、词表大小和高概率 token。
    prune_targets = {3: 34, 5: 24}  # 指定两次渐进裁剪后的目标词表容量。
    for iteration in range(1, iterations + 1):  # 重复执行 E 步、M 步和可选裁剪。
        expected_counts = Counter()  # 创建全语料 token 后验期望计数器。
        corpus_log_likelihood = 0.0  # 初始化按业务频次加权的语料对数似然。
        for text, frequency in rows:  # 遍历每条聚合查询。
            partition, expected, _ = forward_backward(text, probabilities)  # 对当前查询求全部路径概率和 token 后验。
            corpus_log_likelihood += frequency * math.log(partition)  # 按真实出现次数累加对数似然。
            for piece, value in expected.items():  # 遍历当前查询中每个 token 的期望次数。
                expected_counts[piece] += frequency * value  # 按业务频次累加到全语料 E 步统计。
        smoothed_counts = {piece: expected_counts.get(piece, 0.0) + 1e-8 for piece in probabilities}  # 给当前词表每项加入极小伪计数以避免浮点下溢切断保底路径。
        normalizer = sum(smoothed_counts.values())  # 计算带数值保护的 M 步概率归一化常数。
        probabilities = {piece: count / normalizer for piece, count in smoothed_counts.items()}  # 用平滑后的期望次数更新每个 token 概率。
        if iteration in prune_targets and len(probabilities) > prune_targets[iteration]:  # 到达裁剪轮次且词表仍超预算时执行裁剪。
            target_size = prune_targets[iteration]  # 读取本阶段允许保留的词表容量。
            optional = [(piece, probability) for piece, probability in probabilities.items() if piece not in mandatory]  # 分离允许删除的多字符候选。
            optional = sorted(optional, key=lambda item: (-item[1], item[0]))  # 按概率降序和字典序稳定选择保留项。
            keep_optional = {piece for piece, _ in optional[: max(0, target_size - len(mandatory))]}  # 计算容量扣除保底字符后的可选名额。
            keep = set(mandatory) | keep_optional  # 合并保底字符和高概率多字符候选。
            probabilities = {piece: probability for piece, probability in probabilities.items() if piece in keep}  # 删除低概率可选 token。
            retained_total = sum(probabilities.values())  # 计算裁剪后剩余概率质量。
            probabilities = {piece: probability / retained_total for piece, probability in probabilities.items()}  # 重新归一化裁剪后的概率。
        top_pieces = sorted(probabilities.items(), key=lambda item: (-item[1], item[0]))[:6]  # 提取本轮最高概率 token 供观察。
        history.append((iteration, corpus_log_likelihood, len(probabilities), top_pieces))  # 保存训练轨迹中的关键状态。
    return probabilities, history  # 返回最终词表概率和完整 EM 轨迹。
unigram_probabilities, em_history = train_unigram(query_frequencies, initial_probabilities, mandatory_characters)  # 执行八轮可复现 Unigram 训练。
print("轮次 | 加权log-likelihood | 词表数 | 概率最高的 token")  # 输出 EM 与裁剪轨迹表标题。
for iteration, likelihood, vocabulary_size, top_pieces in em_history:  # 遍历并展示每轮完整训练状态。
    top_view = ", ".join(f"{piece}:{probability:.3f}" for piece, probability in top_pieces)  # 格式化高概率 token 便于横向比较。
    print(f"{iteration:>4} | {likelihood:>18.3f} | {vocabulary_size:>6} | {top_view}")  # 输出似然、容量和 token 概率。

轮次 | 加权log-likelihood | 词表数 | 概率最高的 token
   1 |          -3618.405 |     59 | 机器学习:0.156, 深度学习:0.136, 搜索引擎:0.116, 机器翻译:0.090, 机器视觉:0.085, 学习资料:0.080
   2 |          -1932.651 |     59 | 机器学习:0.207, 深度学习:0.173, 搜索引擎:0.139, 机器翻译:0.110, 机器视觉:0.104, 学习资料:0.098
   3 |          -1744.588 |     34 | 机器学习:0.209, 深度学习:0.174, 搜索引擎:0.140, 机器翻译:0.110, 机器视觉:0.105, 学习资料:0.099
   4 |          -1739.769 |     34 | 机器学习:0.209, 深度学习:0.174, 搜索引擎:0.140, 机器翻译:0.110, 机器视觉:0.105, 学习资料:0.099
   5 |          -1739.762 |     24 | 机器学习:0.250, 深度学习:0.208, 搜索引擎:0.167, 机器翻译:0.132, 机器视觉:0.125, 学习资料:0.118
   6 |         -15260.970 |     24 | 机器学习:0.141, 深度学习:0.117, 搜索引擎:0.094, 机器翻译:0.074, 机器视觉:0.070, 学习资料:0.066
   7 |          -3301.256 |     24 | 机器学习:0.141, 深度学习:0.117, 搜索引擎:0.094, 机器翻译:0.074, 机器视觉:0.070, 学习资料:0.066
   8 |          -3301.256 |     24 | 机器学习:0.141, 深度学习:0.117, 搜索引擎:0.094, 机器翻译:0.074, 机器视觉:0.070, 学习资料:0.066


## Viterbi 编码与结果表

把每个字符位置看作节点、每个候选 token 看作一条带 `-log p(token)` 代价的边，问题就变成有向无环图最短路。下面打印新组合查询的最优切分及路径代价，并与字符基线比较。

In [5]:
def viterbi_encode(text, probabilities, max_piece_length=4):  # 用动态规划寻找总负对数概率最小的切分路径。
    length = len(text)  # 保存输入字符长度以创建状态数组。
    costs = [math.inf] * (length + 1)  # 初始化到每个字符位置的最小累计代价。
    costs[0] = 0.0  # 空前缀的路径代价为零。
    previous = [None] * (length + 1)  # 保存最优路径上每个位置的前驱和 token。
    for start in range(length):  # 按字符位置的拓扑顺序扩展状态。
        if not math.isfinite(costs[start]):  # 不可达位置不能产生合法后续边。
            continue  # 跳过当前不可达位置。
        for end in range(start + 1, min(length, start + max_piece_length) + 1):  # 枚举从当前位置出发的候选 token。
            piece = text[start:end]  # 取得候选 token 的表面字符串。
            if piece not in probabilities:  # 已裁剪 token 不属于当前 lattice。
                continue  # 跳过不存在的候选边。
            candidate_cost = costs[start] - math.log(probabilities[piece])  # 计算经当前 token 到达右端点的累计负对数概率。
            if candidate_cost < costs[end]:  # 仅在新路径概率更高时更新最优状态。
                costs[end] = candidate_cost  # 保存更小的累计路径代价。
                previous[end] = (start, piece)  # 保存回溯所需的前驱位置与 token。
    tokens = []  # 创建列表逆序收集最优路径 token。
    position = length  # 从句尾状态开始沿前驱回溯。
    while position > 0:  # 持续回溯直到到达空前缀。
        start, piece = previous[position]  # 读取当前位置最优边的起点和 token。
        tokens.append(piece)  # 逆序保存当前最优 token。
        position = start  # 移动到当前边的起点继续回溯。
    tokens.reverse()  # 把逆序回溯结果恢复为原文本顺序。
    return tokens, costs[length], costs  # 返回最优切分、总成本和全部前缀成本。
evaluation_queries = ["机器学习", "机器搜索", "深度视觉", "搜索优化", "机器学习资料", "引擎学习"]  # 构造训练内查询和未见组合的统一评测集。
print("查询           字符数  Unigram数  路径成本   Unigram切分")  # 输出逐样本结果对照表标题。
unigram_results = {}  # 保存评测查询的切分和代价供后续分析。
for query in evaluation_queries:  # 对每个训练内或组合查询执行同一概率模型。
    tokens, cost, prefix_costs = viterbi_encode(query, unigram_probabilities)  # 运行 Viterbi 得到全局最优路径。
    unigram_results[query] = (tokens, cost)  # 保存结果供失败案例和测试复用。
    print(f"{query:<12} {len(query):>6} {len(tokens):>10} {cost:>9.3f}  {' | '.join(tokens)}")  # 输出长度、成本和真实 token 边界。
unigram_average = weighted_average_length(query_frequencies, lambda text: viterbi_encode(text, unigram_probabilities)[0])  # 计算训练分布上的加权平均 token 数。
print(f"加权平均 token 数：字符基线={baseline_average:.3f}，Unigram={unigram_average:.3f}，减少={1.0 - unigram_average / baseline_average:.1%}")  # 输出同口径压缩结果。

查询           字符数  Unigram数  路径成本   Unigram切分
机器学习              4          1     1.962  机器学习
机器搜索              4          4    56.825  机 | 器 | 搜 | 索
深度视觉              4          4    56.825  深 | 度 | 视 | 觉
搜索优化              4          4    11.635  搜 | 索 | 优 | 化
机器学习资料            6          3    53.112  机器学习 | 资 | 料
引擎学习              4          4    57.111  引 | 擎 | 学 | 习
加权平均 token 数：字符基线=4.000，Unigram=1.488，减少=62.8%


## 结果解读

模型给“机器”“学习”“深度”“搜索”等跨查询复用的片段较高概率，新组合“机器搜索”即使没完整出现，也能由已有 token 拼成。EM 早期在冗余 lattice 中分配软后验，裁剪后词表容量下降，随后继续迭代让剩余 token 重新竞争概率质量。压缩只是可观测结果之一；真正训练目标是语料似然，不能为了少一个 token 强行保留低概率长片段。

## 失败案例：最长优先会选择低概率长 token

若词表同时包含“机器学”“机器”“学习”和“习”，最长匹配会先选低概率的“机器学”，而 Viterbi 会比较整条路径的联合概率。这个最小反例直接展示为什么 Unigram 编码不能复用 WordPiece 的贪心策略。

In [6]:
toy_probabilities = {"机器学": 0.01, "习": 0.01, "机器": 0.20, "学习": 0.20, "机": 0.02, "器": 0.02, "学": 0.02}  # 构造长 token 概率反而较低的可复现词表。
def greedy_longest(text, probabilities, max_piece_length=4):  # 定义错误的最长词片优先编码器。
    tokens = []  # 创建列表保存贪心路径。
    position = 0  # 从原文本首字符开始扫描。
    while position < len(text):  # 持续选择直到覆盖完整输入。
        candidates = [text[position:end] for end in range(position + 1, min(len(text), position + max_piece_length) + 1) if text[position:end] in probabilities]  # 收集当前位置全部合法候选。
        chosen = sorted(candidates, key=lambda piece: (-len(piece), piece))[0]  # 错误地只按长度选择候选而忽略概率。
        tokens.append(chosen)  # 保存本轮贪心 token。
        position += len(chosen)  # 移动到被选择 token 的右边界。
    cost = sum(-math.log(probabilities[token]) for token in tokens)  # 事后计算贪心路径的真实负对数概率。
    return tokens, cost  # 返回贪心切分及其路径代价。
greedy_tokens, greedy_cost = greedy_longest("机器学习", toy_probabilities)  # 对反例运行错误的最长匹配。
optimal_tokens, optimal_cost, toy_costs = viterbi_encode("机器学习", toy_probabilities)  # 对同一反例运行全局最优 Viterbi。
print("输入：机器学习")  # 输出失败案例的固定输入。
print("最长优先：", greedy_tokens, f"，负对数代价={greedy_cost:.3f}")  # 展示长 token 导致的低联合概率路径。
print("Viterbi 修复：", optimal_tokens, f"，负对数代价={optimal_cost:.3f}")  # 展示动态规划找到的高概率切分。
print("各前缀最优代价：", ["∞" if not math.isfinite(value) else f"{value:.3f}" for value in toy_costs])  # 输出动态规划中间状态证明修复过程。

输入：机器学习
最长优先： ['机器学', '习'] ，负对数代价=9.210
Viterbi 修复： ['机器', '学习'] ，负对数代价=3.219
各前缀最优代价： ['0.000', '3.912', '1.609', '4.605', '3.219']


## 生产差距与落地清单

教学实现使用短句和普通概率，线上应在 log-space 做 forward-backward，并按“移除某 token 后的似然损失”而不只是当前概率裁剪。还需实现 Unicode 规范化、空白转义、未知字节回退、特殊 token 保护与确定性词表 id；发布时监控语言分组的 token/character 比、解码可逆率、P99 延迟以及下游任务变化。若启用 subword regularization，训练时可按路径后验采样，但推理和缓存键必须明确是否固定为 Viterbi 结果。

## 最小回归测试

断言只保护概率归一化、保底覆盖和全局最优性，真正的机制证据来自 EM 轨迹、切分表和贪心失败路径。

In [7]:
assert abs(sum(unigram_probabilities.values()) - 1.0) < 1e-9  # 验证最终 Unigram token 概率严格归一化。
assert mandatory_characters.issubset(unigram_probabilities.keys())  # 验证所有训练字符在裁剪后仍有保底路径。
assert all("".join(tokens) == query for query, (tokens, _) in unigram_results.items())  # 验证评测查询的 Viterbi 切分可完整还原原文。
assert unigram_average < baseline_average  # 验证同一训练分布上 Unigram 相对字符基线减少 token 数。
assert optimal_cost < greedy_cost  # 验证反例中动态规划路径的联合概率严格优于最长匹配。
assert optimal_tokens == ["机器", "学习"]  # 验证 Viterbi 选择预期的两个高概率语义片段。
print("最小回归测试通过：EM 概率、字符保底、切分可逆与 Viterbi 全局最优均符合预期。")  # 输出完整执行成功的明确结论。

最小回归测试通过：EM 概率、字符保底、切分可逆与 Viterbi 全局最优均符合预期。
